# Generation and Sampling

## Learning Objectives

In the previous notebook, we trained a tiny next-token model.

Given a context such as:

```text
["sat", "on", "the"]
```

the model learned a probability distribution over the vocabulary.

For example, it learned approximately:

```text
P("mat" | "sat on the") ≈ 0.5
P("rug" | "sat on the") ≈ 0.5
```

During training, the next token was already known.

We could compare the model's prediction with the observed target and calculate cross-entropy loss.

Generation is different.

During generation:

```text
there is no target token
```

The model produces a distribution, and we must decide how to choose a token from it.

This notebook will build that process directly.

We will study:

- the difference between training and generation
- greedy decoding
- probability-based sampling
- autoregressive generation
- why generated tokens become future inputs
- temperature
- how temperature changes a probability distribution
- deterministic vs stochastic generation
- the effect of context length
- limitations of our simple fixed-context model

The key transition is:

```text
training:
context + known target
→ loss
→ parameter update

generation:
context
→ probability distribution
→ choose token
→ append token
→ repeat
```

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


## 1. Recreate the Training Corpus

This notebook is self-contained.

We will train the same tiny next-token model from the previous notebook before using it for generation.

Our corpus is:

```text
the cat sat on the mat
the cat ate the fish
the dog sat on the rug
```

The model will use:

```text
context length T = 3
embedding dimension C = 4
```

These remain intentionally small values so that every part of the model can be inspected.

In [2]:
sentences = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
]

tokenized_sentences = [
    sentence.split()
    for sentence in sentences
]

all_tokens = [
    token
    for sentence_tokens in tokenized_sentences
    for token in sentence_tokens
]

vocab = sorted(set(all_tokens))

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

vocab_size = len(vocab)

print("Vocabulary:")
print(vocab)

print("\nVocabulary size:", vocab_size)

print("\nToken → ID:")
print(token_to_id)

Vocabulary:
['ate', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']

Vocabulary size: 9

Token → ID:
{'ate': 0, 'cat': 1, 'dog': 2, 'fish': 3, 'mat': 4, 'on': 5, 'rug': 6, 'sat': 7, 'the': 8}


## 2. Encode the Corpus

The model cannot operate directly on strings.

We therefore convert each token into its integer vocabulary ID.

The transformation is:

```text
text
↓
tokens
↓
token IDs
```

We keep each sentence separate so that context windows do not accidentally cross sentence boundaries.

In [3]:
encoded_sentences = [
    torch.tensor(
        [token_to_id[token] for token in sentence_tokens],
        dtype=torch.long,
    )
    for sentence_tokens in tokenized_sentences
]

for sentence, encoded in zip(
    sentences,
    encoded_sentences,
):
    print(sentence)
    print(encoded)
    print()

the cat sat on the mat
tensor([8, 1, 7, 5, 8, 4])

the cat ate the fish
tensor([8, 1, 0, 8, 3])

the dog sat on the rug
tensor([8, 2, 7, 5, 8, 6])



## 3. Create Next-Token Training Examples

We use:

```text
context length T = 3
```

so each training example contains:

```text
3 input tokens
→
1 next-token target
```

For example:

```text
the cat sat → on
```

The model learns:

$$
P(\text{next token}\mid\text{previous 3 tokens})
$$

Our architecture has a fixed context length, so every input contains exactly three tokens.

In [4]:
context_length = 3

contexts = []
targets = []

for sentence_ids in encoded_sentences:
    for i in range(
        len(sentence_ids) - context_length
    ):
        contexts.append(
            sentence_ids[
                i:i + context_length
            ]
        )

        targets.append(
            sentence_ids[
                i + context_length
            ]
        )

contexts = torch.stack(contexts)
targets = torch.stack(targets)

print("Contexts shape:", contexts.shape)
print("Targets shape:", targets.shape)

Contexts shape: torch.Size([8, 3])
Targets shape: torch.Size([8])


In [5]:
for context_ids, target_id in zip(
    contexts,
    targets,
):
    context_tokens = [
        id_to_token[token_id.item()]
        for token_id in context_ids
    ]

    target_token = id_to_token[
        target_id.item()
    ]

    print(
        context_tokens,
        "→",
        target_token,
    )

['the', 'cat', 'sat'] → on
['cat', 'sat', 'on'] → the
['sat', 'on', 'the'] → mat
['the', 'cat', 'ate'] → the
['cat', 'ate', 'the'] → fish
['the', 'dog', 'sat'] → on
['dog', 'sat', 'on'] → the
['sat', 'on', 'the'] → rug


## 4. Recreate the Next-Token Model

Our model has two learnable components.

### Embedding table

```text
token IDs
[B, T]

↓

embedding

[B, T, C]
```

with:

```text
C = 4
```

### Prediction layer

We flatten the three token embeddings:

```text
[B, T, C]

↓

[B, T*C]
```

and use a linear layer to produce one logit for every vocabulary token:

```text
[B, T*C]

↓

Linear(T*C, V)

↓

[B, V]
```

With:

```text
T = 3
C = 4
V = 9
```

the model performs:

```text
[B, 3, 4]
↓
[B, 12]
↓
[B, 9]
```

This is still a deliberately simple model.

Later, Transformers will replace this crude context-flattening mechanism with attention.

In [6]:
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)

prediction_layer = nn.Linear(
    context_length * embedding_dim,
    vocab_size,
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    list(embedding.parameters())
    + list(prediction_layer.parameters()),
    lr=0.1,
)

print(
    "Embedding table shape:",
    embedding.weight.shape,
)

print(
    "Prediction weight shape:",
    prediction_layer.weight.shape,
)

Embedding table shape: torch.Size([9, 4])
Prediction weight shape: torch.Size([9, 12])


## 5. Train the Model

We will train for 500 epochs, just as before.

Each optimization step performs:

```text
clear gradients
↓
embedding lookup
↓
flatten context
↓
compute logits
↓
cross-entropy loss
↓
backpropagation
↓
parameter update
```

The dataset is tiny enough that all eight examples can be processed as one batch.

In [7]:
num_epochs = 500

loss_history = []

for epoch in range(num_epochs):
    optimizer.zero_grad()

    context_embeddings = embedding(
        contexts
    )

    flattened_contexts = (
        context_embeddings.reshape(
            context_embeddings.shape[0],
            -1,
        )
    )

    logits = prediction_layer(
        flattened_contexts
    )

    loss = criterion(
        logits,
        targets,
    )

    loss.backward()
    optimizer.step()

    loss_history.append(
        loss.item()
    )

    if epoch % 100 == 0:
        print(
            f"epoch {epoch:3d} "
            f"loss = {loss.item():.6f}"
        )

epoch   0 loss = 2.486645
epoch 100 loss = 0.206125
epoch 200 loss = 0.187567
epoch 300 loss = 0.182143
epoch 400 loss = 0.179617


In [8]:
print(
    "Initial loss:",
    loss_history[0],
)

print(
    "Final loss:",
    loss_history[-1],
)

Initial loss: 2.4866445064544678
Final loss: 0.17818431556224823


## 6. Create a Next-Token Prediction Function

During generation we will repeatedly perform the same operation:

```text
context
↓
token IDs
↓
embedding
↓
flatten
↓
prediction layer
↓
logits
```

Instead of rewriting this sequence every time, we will create a small helper function.

The function will accept a tensor containing exactly:

```text
T = 3
```

token IDs and return the model's logits over all:

```text
V = 9
```

possible next tokens.

This is our model's basic inference operation:

```text
context
→ logits over next token
```

In [9]:
def get_next_token_logits(
    context_ids,
):
    """
    context_ids:
        tensor of shape [T]

    returns:
        logits of shape [V]
    """

    context_batch = (
        context_ids.unsqueeze(0)
    )

    context_embeddings = embedding(
        context_batch
    )

    flattened_context = (
        context_embeddings.reshape(
            1,
            -1,
        )
    )

    logits = prediction_layer(
        flattened_context
    )

    return logits[0]

## 7. Why Add a Batch Dimension?

Our helper receives one context with shape:

```text
[T]
```

For example:

```text
[8, 1, 7]
```

But our layers were used during training with a batch dimension:

```text
[B, T]
```

For one example we therefore create:

```text
[1, T]
```

using:

```python
context_ids.unsqueeze(0)
```

So:

```text
[T]
↓
unsqueeze(0)
↓
[1, T]
```

After prediction:

```text
[1, V]
```

we select the first and only batch item:

```python
logits[0]
```

giving:

```text
[V]
```

This means our helper returns one score for every possible next token.

In [10]:
test_context_tokens = [
    "sat",
    "on",
    "the",
]

test_context_ids = torch.tensor(
    [
        token_to_id[token]
        for token in test_context_tokens
    ],
    dtype=torch.long,
)

with torch.no_grad():
    test_logits = get_next_token_logits(
        test_context_ids
    )

    test_probabilities = torch.softmax(
        test_logits,
        dim=-1,
    )

print("Context:")
print(test_context_tokens)

print("\nLogits shape:")
print(test_logits.shape)

print("\nProbabilities:")

for token_id, probability in enumerate(
    test_probabilities
):
    print(
        f"{id_to_token[token_id]:>5}: "
        f"{probability.item():.6f}"
    )

Context:
['sat', 'on', 'the']

Logits shape:
torch.Size([9])

Probabilities:
  ate: 0.000684
  cat: 0.000318
  dog: 0.000520
 fish: 0.002131
  mat: 0.497040
   on: 0.001616
  rug: 0.496957
  sat: 0.000628
  the: 0.000106


## 8. Training and Generation Use the Same Model Differently

During training we had:

```text
context
+
known next-token target
```

For example:

```text
["the", "cat", "sat"]
→ target = "on"
```

The target let us calculate:

```text
cross-entropy loss
```

and therefore gradients.

Generation is different.

Suppose we provide:

```text
["sat", "on", "the"]
```

The model outputs a distribution such as:

```text
mat ≈ 0.50
rug ≈ 0.50
other tokens ≈ very small
```

There is no correct target available to us during generation.

Instead, we must make a decision:

> Which token should we actually output?

This decision is called **decoding**.

Different decoding strategies can produce different text from the exact same trained model.

## 9. Greedy Decoding

The simplest decoding strategy is:

> Choose the token with the highest probability.

Because softmax preserves the ordering of logits, we can equivalently choose the largest logit.

Mathematically:

$$
\hat{y}
=
\arg\max_i z_i
$$

where:

```text
z_i = logit for vocabulary token i
```

This is called **greedy decoding**.

For example, suppose the model predicts:

```text
mat  0.4970
rug  0.4969
fish 0.0021
...
```

Greedy decoding chooses:

```text
mat
```

because it has the highest probability, even though `"rug"` is almost equally likely.

Greedy decoding is deterministic:

```text
same model
+
same context
→
same next token
```

as long as the model itself behaves deterministically.

In [11]:
with torch.no_grad():
    logits = get_next_token_logits(
        test_context_ids
    )

    probabilities = torch.softmax(
        logits,
        dim=-1,
    )

    greedy_token_id = (
        torch.argmax(
            probabilities
        )
        .item()
    )

    greedy_token = id_to_token[
        greedy_token_id
    ]

print(
    "Context:",
    test_context_tokens,
)

print(
    "Greedy next token:",
    greedy_token,
)

print(
    "Probability:",
    probabilities[
        greedy_token_id
    ].item(),
)

Context: ['sat', 'on', 'the']
Greedy next token: mat
Probability: 0.49704018235206604


## 10. Greedy Decoding Throws Information Away

Suppose the model predicts:

```text
mat  → 49.704%
rug  → 49.696%
```

Greedy decoding sees only:

```text
49.704% > 49.696%
```

and chooses:

```text
mat
```

The near-tie is discarded.

This is an important distinction between:

```text
model prediction
```

and:

```text
decoding decision
```

The model itself represented substantial uncertainty:

```text
mat ≈ rug
```

but greedy decoding converts that distribution into one deterministic token.

So:

```text
probability distribution
≠
generated token
```

The distribution is produced by the model.

The decoding algorithm determines how we use it.

## 11. From One Next Token to a Sequence

Generating one token is only the first step.

Suppose our starting context is:

```text
the cat sat
```

The model predicts:

```text
on
```

Now we have:

```text
the cat sat on
```

To generate another token, we need another context.

Because our model was trained with:

```text
T = 3
```

we use the most recent three tokens:

```text
cat sat on
```

The model might then predict:

```text
the
```

Now the generated sequence becomes:

```text
the cat sat on the
```

and the next context becomes:

```text
sat on the
```

Then the model predicts again.

So generation proceeds as:

```text
the cat sat
        ↓
       on

cat sat on
        ↓
       the

sat on the
        ↓
       mat

...
```

Each generated token becomes part of the input used to generate later tokens.

This repeated feedback process is called **autoregressive generation**.

## 12. Autoregressive Generation

The word **autoregressive** means that the model uses previously observed or generated outputs when predicting future outputs.

Conceptually:

$$
P(x_1,x_2,\dots,x_n)
=
\prod_t
P(x_t \mid x_{<t})
$$

In words:

```text
probability of an entire sequence
=
product of conditional next-token probabilities
```

Generation therefore repeatedly performs:

```text
current context
↓
predict next-token distribution
↓
choose next token
↓
append token
↓
update context
↓
repeat
```

Our model can only inspect the latest three tokens because:

```text
context length = 3
```

So after every generated token, we keep the most recent three token IDs.

In [14]:
def generate_greedy(
    start_tokens,
    num_new_tokens,
):
    generated_tokens = list(
        start_tokens
    )

    for _ in range(num_new_tokens):
        current_context = (
            generated_tokens[
                -context_length:
            ]
        )

        context_ids = torch.tensor(
            [
                token_to_id[token]
                for token in current_context
            ],
            dtype=torch.long,
        )

        with torch.no_grad():
            logits = get_next_token_logits(
                context_ids
            )

        next_token_id = (
            torch.argmax(logits)
            .item()
        )

        next_token = id_to_token[
            next_token_id
        ]

        generated_tokens.append(
            next_token
        )

    return generated_tokens

## 13. Why Do We Keep Only the Last Three Tokens?

Our model was trained with input shape:

```text
[T]
```

where:

```text
T = 3
```

Its prediction layer expects:

```text
T * C
=
3 * 4
=
12
```

input features.

Therefore, during generation, we must continue giving it exactly three token positions.

If our generated sequence becomes:

```text
the cat sat on the
```

the model does **not** consume all five tokens.

It sees only:

```text
sat on the
```

because that is the most recent context window of length three.

This is an important limitation of our current model.

Information older than three tokens is completely inaccessible.

Later, context length will become a central property of language models and Transformers.

In [15]:
greedy_output = generate_greedy(
    start_tokens=[
        "the",
        "cat",
        "sat",
    ],
    num_new_tokens=6,
)

print(
    "Generated sequence:"
)

print(
    " ".join(
        greedy_output
    )
)

Generated sequence:
the cat sat on the mat cat the fish


## 14. Inspect Generation Step by Step

Seeing only the final generated string hides what the model did at each step.

Let's inspect:

```text
context
→ chosen token
→ chosen-token probability
```

for every generation step.

This will let us see exactly how autoregressive generation evolves.

In [16]:
def generate_greedy_verbose(
    start_tokens,
    num_new_tokens,
):
    generated_tokens = list(
        start_tokens
    )

    for step in range(
        num_new_tokens
    ):
        current_context = (
            generated_tokens[
                -context_length:
            ]
        )

        context_ids = torch.tensor(
            [
                token_to_id[token]
                for token in current_context
            ],
            dtype=torch.long,
        )

        with torch.no_grad():
            logits = get_next_token_logits(
                context_ids
            )

            probabilities = torch.softmax(
                logits,
                dim=-1,
            )

        next_token_id = (
            torch.argmax(
                probabilities
            )
            .item()
        )

        next_token = id_to_token[
            next_token_id
        ]

        next_probability = (
            probabilities[
                next_token_id
            ]
            .item()
        )

        print(
            f"step {step + 1}: "
            f"{current_context} "
            f"→ {next_token} "
            f"(p={next_probability:.4f})"
        )

        generated_tokens.append(
            next_token
        )

    return generated_tokens

In [17]:
greedy_output = (
    generate_greedy_verbose(
        start_tokens=[
            "the",
            "cat",
            "sat",
        ],
        num_new_tokens=8,
    )
)

print("\nFinal sequence:")

print(
    " ".join(
        greedy_output
    )
)

step 1: ['the', 'cat', 'sat'] → on (p=0.9959)
step 2: ['cat', 'sat', 'on'] → the (p=0.9955)
step 3: ['sat', 'on', 'the'] → mat (p=0.4970)
step 4: ['on', 'the', 'mat'] → cat (p=0.3501)
step 5: ['the', 'mat', 'cat'] → the (p=0.3365)
step 6: ['mat', 'cat', 'the'] → fish (p=0.3526)
step 7: ['cat', 'the', 'fish'] → cat (p=0.2843)
step 8: ['the', 'fish', 'cat'] → the (p=0.5245)

Final sequence:
the cat sat on the mat cat the fish cat the


## 15. Interpreting Greedy Autoregressive Generation

The greedy generation trace begins very confidently:

```text
["the", "cat", "sat"] → "on"   P ≈ 0.996
["cat", "sat", "on"]  → "the"  P ≈ 0.996
```

These contexts appeared directly in the training data, so the model learned them very well.

The third context is:

```text
["sat", "on", "the"]
```

which appeared with two different targets:

```text
"mat"
"rug"
```

The learned probabilities are approximately:

```text
P("mat") ≈ 0.4970
P("rug") ≈ 0.4970
```

Greedy decoding chooses `"mat"` only because its probability is microscopically larger.

The generated sequence is now:

```text
the cat sat on the mat
```

### Generation now leaves the training distribution

The next context becomes:

```text
["on", "the", "mat"]
```

But this context was never present as an input example during training.

Our training examples included:

```text
["the", "cat", "sat"]
["cat", "sat", "on"]
["sat", "on", "the"]
["the", "cat", "ate"]
["cat", "ate", "the"]
["the", "dog", "sat"]
["dog", "sat", "on"]
```

but not:

```text
["on", "the", "mat"]
```

The model must therefore make a prediction for an unseen combination of tokens.

Its confidence immediately falls:

```text
["on", "the", "mat"] → "cat"  P ≈ 0.350
```

and later predictions also operate on contexts created by the model itself rather than contexts observed during training.

This leads to:

```text
the cat sat on the mat cat the fish cat the
```

The strange continuation is not surprising.

The model was trained on only eight context-target examples and has no explicit knowledge of grammar, sentence boundaries, or broader language structure.

### Autoregressive generation creates its own future inputs

This illustrates an important property of generation:

```text
model prediction
↓
generated token
↓
becomes part of next input
↓
next prediction
↓
becomes part of another input
↓
...
```

An early generated token can therefore move the model into contexts that were rare or absent during training.

This creates a difference between training and generation:

```text
training
→ contexts come from the dataset

generation
→ future contexts increasingly contain the model's own outputs
```

Our toy model makes this effect especially visible because the dataset is extremely small.

It also demonstrates why low training loss alone does not guarantee good generation.

The model nearly optimized its training objective perfectly, yet long free-running generation quickly becomes poor.

That is not a contradiction:

```text
excellent fit to observed next-token examples
≠
strong generalization to unseen generated contexts
```

## 16. Sampling Instead of Always Taking the Maximum

Greedy decoding always chooses:

```text
argmax(probability)
```

For our ambiguous context:

```text
["sat", "on", "the"]
```

the model predicts approximately:

```text
mat → 0.4970
rug → 0.4970
```

Greedy decoding always chooses `"mat"` because its probability is very slightly larger.

But that throws away the uncertainty learned by the model.

An alternative is **sampling**.

Instead of always taking the highest-probability token, we randomly choose a token according to the predicted probability distribution.

Conceptually, if:

```text
mat  → 50%
rug  → 50%
```

then repeated sampling should produce roughly:

```text
mat about half the time
rug about half the time
```

Sampling therefore allows multiple possible continuations from the same model and context.

In PyTorch, we can sample from a categorical distribution with:

```python
torch.multinomial(probabilities, num_samples=1)
```

In [18]:
with torch.no_grad():
    logits = get_next_token_logits(
        test_context_ids
    )

    probabilities = torch.softmax(
        logits,
        dim=-1,
    )

torch.manual_seed(42)

print("Repeated samples:")

for _ in range(20):
    sampled_id = torch.multinomial(
        probabilities,
        num_samples=1,
    ).item()

    print(
        id_to_token[sampled_id],
        end=" "
    )

Repeated samples:
mat rug rug mat rug rug rug rug rug rug rug rug mat mat rug mat rug rug rug rug 

## 17. Sampling Is Stochastic

Unlike greedy decoding:

```text
same context
→ same token
```

sampling can produce:

```text
same context
→ different tokens on different runs
```

The randomness is not arbitrary.

Tokens are selected according to the probability distribution learned by the model.

A token with:

```text
P = 0.50
```

is far more likely to be selected than a token with:

```text
P = 0.001
```

So sampling preserves the model's uncertainty instead of collapsing it immediately to one argmax decision.

Because random sampling is involved, we use:

```python
torch.manual_seed(...)
```

when we want an experiment to be reproducible.

Changing the seed can change the sampled sequence even though the trained model and its probabilities remain unchanged.

In [19]:
torch.manual_seed(42)

num_samples = 10_000

samples = torch.multinomial(
    probabilities,
    num_samples=num_samples,
    replacement=True,
)

counts = torch.bincount(
    samples,
    minlength=vocab_size,
)

print(
    f"Empirical frequencies from "
    f"{num_samples:,} samples:\n"
)

for token_id in range(vocab_size):
    empirical_probability = (
        counts[token_id].item()
        / num_samples
    )

    print(
        f"{id_to_token[token_id]:>5}: "
        f"model={probabilities[token_id].item():.4f} "
        f"sampled={empirical_probability:.4f}"
    )

Empirical frequencies from 10,000 samples:

  ate: model=0.0007 sampled=0.0006
  cat: model=0.0003 sampled=0.0004
  dog: model=0.0005 sampled=0.0007
 fish: model=0.0021 sampled=0.0014
  mat: model=0.4970 sampled=0.4936
   on: model=0.0016 sampled=0.0016
  rug: model=0.4970 sampled=0.5005
  sat: model=0.0006 sampled=0.0010
  the: model=0.0001 sampled=0.0002


## 18. Autoregressive Generation with Sampling

Greedy generation always chooses the highest-probability token.

Sampling instead draws from the entire next-token distribution.

This changes the generation loop from:

```text
context
↓
predict distribution
↓
argmax
↓
append token
↓
repeat
```

to:

```text
context
↓
predict distribution
↓
sample from distribution
↓
append sampled token
↓
repeat
```

The model itself has not changed.

Only the decoding strategy has changed.

This means two generation runs can start from the same context and produce different sequences.

In [20]:
def generate_sampled(
    start_tokens,
    num_new_tokens,
    seed=None,
):
    if seed is not None:
        torch.manual_seed(seed)

    generated_tokens = list(start_tokens)

    for _ in range(num_new_tokens):
        current_context = generated_tokens[
            -context_length:
        ]

        context_ids = torch.tensor(
            [
                token_to_id[token]
                for token in current_context
            ],
            dtype=torch.long,
        )

        with torch.no_grad():
            logits = get_next_token_logits(
                context_ids
            )

            probabilities = torch.softmax(
                logits,
                dim=-1,
            )

        next_token_id = torch.multinomial(
            probabilities,
            num_samples=1,
        ).item()

        next_token = id_to_token[
            next_token_id
        ]

        generated_tokens.append(
            next_token
        )

    return generated_tokens

In [21]:
for seed in [1, 2, 3, 4, 5]:
    output = generate_sampled(
        start_tokens=[
            "the",
            "cat",
            "sat",
        ],
        num_new_tokens=8,
        seed=seed,
    )

    print(
        f"seed {seed}:",
        " ".join(output),
    )

seed 1: the cat sat on the rug the mat ate the fish
seed 2: the cat sat on the rug ate the fish ate the
seed 3: the cat sat on the mat on ate the fish fish
seed 4: the cat sat on the mat ate the fish mat on
seed 5: the cat sat on the mat the mat cat on the


## 19. Why Sampled Sequences Can Diverge Quickly

The first few contexts may be highly predictable:

```text
the cat sat → on
cat sat on  → the
```

But once generation reaches:

```text
sat on the
```

the model has roughly two equally likely options:

```text
mat
rug
```

If one run samples:

```text
mat
```

and another samples:

```text
rug
```

their next contexts become different:

```text
on the mat
```

versus:

```text
on the rug
```

From that point onward, the model receives different inputs and the two generated sequences can continue to diverge.

This is why a single stochastic token choice can affect many later tokens.

Autoregressive generation compounds earlier decisions because generated tokens become future inputs.

## 20. Temperature

Sampling gives us randomness, but sometimes we want to control how strongly sampling favors high-probability tokens.

This is where **temperature** is used.

Before softmax, we divide the logits by a positive temperature value:

$$
P_i(T)
=
\frac{
\exp(z_i / T)
}{
\sum_j \exp(z_j / T)
}
$$

where:

```text
z_i = original logit
T   = temperature
```

Temperature does not change the trained parameters.

It modifies the probability distribution used during decoding.

Three useful regimes are:

```text
T = 1
→ original model distribution

T < 1
→ sharper distribution
→ high-logit tokens become more dominant

T > 1
→ flatter distribution
→ lower-logit tokens receive more probability
```

Temperature acts on **logits before softmax**, not on probabilities after softmax.

## 21. Why Dividing Logits Changes the Distribution

Suppose two tokens have logits:

```text
token A → 4
token B → 2
```

At:

```text
T = 1
```

softmax compares:

```text
exp(4)
vs
exp(2)
```

At a lower temperature such as:

```text
T = 0.5
```

we divide logits by `0.5`:

```text
4 / 0.5 = 8
2 / 0.5 = 4
```

The difference becomes much larger before exponentiation:

```text
exp(8)
vs
exp(4)
```

so the distribution becomes sharper.

At a higher temperature such as:

```text
T = 2
```

we get:

```text
4 / 2 = 2
2 / 2 = 1
```

The logits move closer together, so the resulting probabilities become flatter.

Therefore:

```text
lower temperature
→ amplify relative logit differences

higher temperature
→ reduce relative logit differences
```

In [22]:
temperatures = [
    0.25,
    0.5,
    1.0,
    2.0,
    5.0,
]

with torch.no_grad():
    logits = get_next_token_logits(
        test_context_ids
    )

for temperature in temperatures:
    probabilities_t = torch.softmax(
        logits / temperature,
        dim=-1,
    )

    print(
        f"\nTemperature = {temperature}"
    )

    sorted_probs, sorted_ids = torch.sort(
        probabilities_t,
        descending=True,
    )

    for token_id, probability in zip(
        sorted_ids[:5],
        sorted_probs[:5],
    ):
        print(
            f"{id_to_token[token_id.item()]:>5}: "
            f"{probability.item():.6f}"
        )


Temperature = 0.25
  mat: 0.500168
  rug: 0.499832
 fish: 0.000000
   on: 0.000000
  ate: 0.000000

Temperature = 0.5
  mat: 0.500076
  rug: 0.499907
 fish: 0.000009
   on: 0.000005
  ate: 0.000001

Temperature = 1.0
  mat: 0.497040
  rug: 0.496957
 fish: 0.002131
   on: 0.001616
  ate: 0.000684

Temperature = 2.0
  mat: 0.441049
  rug: 0.441012
 fish: 0.028876
   on: 0.025152
  ate: 0.016365

Temperature = 5.0
  mat: 0.259557
  rug: 0.259548
 fish: 0.087227
   on: 0.082540
  ate: 0.069503


## 22. An Important Subtlety in Our `mat` / `rug` Example

Our two highest logits for:

```text
["sat", "on", "the"]
```

are already almost identical.

Therefore lowering temperature may make the distribution strongly suppress unrelated tokens while still leaving:

```text
mat ≈ rug
```

close to a 50/50 split.

Temperature cannot create a large preference between two logits that are nearly identical.

It only rescales the differences that already exist.

So in this example we expect:

```text
low temperature
→ almost all probability concentrated on mat and rug
→ mat and rug still close to each other

high temperature
→ probability spreads toward the other vocabulary tokens
```

This distinction is important:

> Temperature changes the sharpness of the model's existing preferences; it does not invent new preferences.

In [23]:
def generate_with_temperature(
    start_tokens,
    num_new_tokens,
    temperature=1.0,
    seed=None,
):
    if temperature <= 0:
        raise ValueError(
            "temperature must be > 0"
        )

    if seed is not None:
        torch.manual_seed(seed)

    generated_tokens = list(
        start_tokens
    )

    for _ in range(
        num_new_tokens
    ):
        current_context = (
            generated_tokens[
                -context_length:
            ]
        )

        context_ids = torch.tensor(
            [
                token_to_id[token]
                for token in current_context
            ],
            dtype=torch.long,
        )

        with torch.no_grad():
            logits = get_next_token_logits(
                context_ids
            )

            probabilities = torch.softmax(
                logits / temperature,
                dim=-1,
            )

        next_token_id = torch.multinomial(
            probabilities,
            num_samples=1,
        ).item()

        next_token = id_to_token[
            next_token_id
        ]

        generated_tokens.append(
            next_token
        )

    return generated_tokens

In [24]:
for temperature in [
    0.25,
    0.5,
    1.0,
    2.0,
    5.0,
]:
    output = generate_with_temperature(
        start_tokens=[
            "the",
            "cat",
            "sat",
        ],
        num_new_tokens=10,
        temperature=temperature,
        seed=42,
    )

    print(
        f"T={temperature:<4}:",
        " ".join(output),
    )

T=0.25: the cat sat on the rug the on the rug the rug the
T=0.5 : the cat sat on the rug the on the rug the rug the
T=1.0 : the cat sat on the rug dog on the rug cat cat the
T=2.0 : the cat sat on the rug dog on the rug rug fish sat
T=5.0 : the cat sat fish sat rug dog cat dog rug rug fish sat


## 23. Interpreting Temperature in the Actual Model

The temperature experiment shows two different effects.

### Low temperature suppresses weak alternatives

For:

```text
["sat", "on", "the"]
```

the original model distribution at `T = 1` was approximately:

```text
mat  0.4970
rug  0.4970

all other tokens combined ≈ 0.006
```

At:

```text
T = 0.25
```

the distribution became approximately:

```text
mat  0.5002
rug  0.4998
others ≈ 0
```

The model therefore becomes much less likely to sample tokens it considered weak alternatives.

However, `"mat"` does not become overwhelmingly more likely than `"rug"`.

Their original logits were already almost identical.

Temperature rescales existing logit differences; it does not create a preference that the model did not already have.

---

### High temperature flattens the distribution

At:

```text
T = 5
```

the distribution became much flatter:

```text
mat   ≈ 0.260
rug   ≈ 0.260
fish  ≈ 0.087
on    ≈ 0.083
ate   ≈ 0.070
...
```

The top two tokens are still preferred, but lower-ranked alternatives now receive substantial probability.

This makes surprising tokens much more likely to be sampled.

---

### Generation compounds the effect

Using the same random seed, the generated sequences changed substantially:

```text
T = 0.25
the cat sat on the rug the on the rug the rug the

T = 1.0
the cat sat on the rug dog on the rug cat cat the

T = 5.0
the cat sat fish sat rug dog cat dog rug rug fish sat
```

At low temperature, sampling stays closer to the model's strongest preferences.

At high temperature, lower-probability tokens become much easier to select.

Once one of those tokens is generated, it becomes part of the next context:

```text
sampled token
↓
new context
↓
new probability distribution
↓
next sampled token
```

So a temperature-induced choice at one step can change the entire later generation trajectory.

The useful mental model is:

```text
model parameters
→ determine logits

temperature
→ reshapes those logits for decoding

sampling
→ chooses from the resulting distribution
```

Temperature does not make the model more knowledgeable or change what it learned.

It changes how aggressively decoding follows the model's learned preferences.

## 24. Is Temperature Zero the Same as Greedy Decoding?

We often hear:

```text
temperature = 0
```

used informally to mean deterministic or greedy generation.

But our temperature formula is:

$$
P_i(T)
=
\frac{\exp(z_i/T)}
{\sum_j \exp(z_j/T)}
$$

Setting:

```text
T = 0
```

would require division by zero, so it is not mathematically valid in this formula.

Instead, consider what happens as:

$$
T \rightarrow 0^+
$$

Dividing logits by an increasingly small positive number magnifies their differences.

If one token has a uniquely largest logit, its softmax probability approaches:

```text
1
```

while the other probabilities approach:

```text
0
```

Sampling from that limiting distribution therefore behaves like greedy decoding.

But there is an important distinction:

```text
greedy decoding
→ explicitly choose argmax(logits)

very small temperature + sampling
→ create an extremely sharp distribution and sample from it
```

They can behave nearly identically when there is a clear winner, but they are not the same computation.

Our `"mat"` / `"rug"` example is particularly useful because the two logits are nearly tied.

Even at:

```text
T = 0.25
```

we still observed approximately:

```text
mat ≈ 0.5002
rug ≈ 0.4998
```

A sufficiently low positive temperature would eventually amplify even their tiny difference, but greedy decoding chooses `"mat"` immediately because it only asks which logit is largest.

In [25]:
temperatures = [
    1.0,
    0.5,
    0.25,
    0.1,
    0.05,
    0.01,
]

with torch.no_grad():
    logits = get_next_token_logits(
        test_context_ids
    )

for temperature in temperatures:
    probabilities_t = torch.softmax(
        logits / temperature,
        dim=-1,
    )

    mat_probability = probabilities_t[
        token_to_id["mat"]
    ].item()

    rug_probability = probabilities_t[
        token_to_id["rug"]
    ].item()

    print(
        f"T={temperature:<4} "
        f"mat={mat_probability:.6f} "
        f"rug={rug_probability:.6f}"
    )

T=1.0  mat=0.497040 rug=0.496957
T=0.5  mat=0.500076 rug=0.499907
T=0.25 mat=0.500168 rug=0.499832
T=0.1  mat=0.500421 rug=0.499579
T=0.05 mat=0.500841 rug=0.499159
T=0.01 mat=0.504211 rug=0.495789


## 25. What Happens as Temperature Approaches Zero?

Our experiment produced:

```text
T=1.00   mat=0.497040   rug=0.496957
T=0.50   mat=0.500076   rug=0.499907
T=0.25   mat=0.500168   rug=0.499832
T=0.10   mat=0.500421   rug=0.499579
T=0.05   mat=0.500841   rug=0.499159
T=0.01   mat=0.504211   rug=0.495789
```

Lowering temperature strongly suppresses the lower-ranked vocabulary tokens.

However, `"mat"` and `"rug"` remain close to a 50/50 split even at very low temperature.

Why?

Temperature rescales logits:

$$
z_i
\rightarrow
\frac{z_i}{T}
$$

For two tokens, what matters is their logit difference:

$$
\Delta z
=
z_{\text{mat}}
-
z_{\text{rug}}
$$

After temperature scaling, that difference becomes:

$$
\frac{\Delta z}{T}
$$

If the original logit difference is extremely small, lowering temperature initially does little to separate the two tokens.

This is exactly what happened here.

The model learned almost no preference between:

```text
"mat"
```

and:

```text
"rug"
```

because the training data provided one occurrence of each continuation for the same context.

As temperature decreases:

```text
weak alternatives
→ rapidly lose probability

mat vs rug
→ remain nearly tied
```

At `T = 0.01`, almost all probability mass belongs to those two tokens, but the split is still only approximately:

```text
50.42% mat
49.58% rug
```

### Greedy decoding is still different

Greedy decoding does not care how small the difference is.

It computes:

```text
argmax(logits)
```

so even a microscopic advantage is enough to choose `"mat"` every time.

Sampling at a small positive temperature instead preserves some randomness until the distribution becomes extremely concentrated.

Therefore:

```text
greedy decoding
≠ literally temperature 0
```

A better mental model is:

```text
T → 0+
→ softmax becomes increasingly concentrated on the largest logit

greedy
→ directly choose the largest logit
```

When there is a clear winner, these behave nearly the same.

When logits are extremely close, as in this experiment, the distinction becomes much easier to see.

# What We Learned

This notebook connected a trained next-token model to text generation.

## 1. A language model produces a distribution, not a word

For a context, the model produces:

```text
logits
[V]
```

which softmax can convert into:

```text
probabilities
[V]
```

The model itself does not inherently decide which token must be emitted.

Decoding determines how that distribution is used.

---

## 2. Greedy decoding chooses the largest logit

Greedy decoding performs:

```text
argmax(logits)
```

and is deterministic for a fixed model and context.

For the ambiguous context:

```text
["sat", "on", "the"]
```

our model learned approximately:

```text
mat ≈ 0.497
rug ≈ 0.497
```

but greedy decoding always selected `"mat"` because its logit was microscopically larger.

This shows that:

```text
model uncertainty
≠
decoding behavior
```

Greedy decoding can hide substantial uncertainty in the underlying model distribution.

---

## 3. Generation is autoregressive

After choosing a token, we append it to the generated sequence and use recent tokens to predict again:

```text
context
↓
next-token distribution
↓
choose token
↓
append
↓
new context
↓
repeat
```

Generated outputs therefore become future inputs.

This feedback loop is autoregressive generation.

---

## 4. Generation can leave the training distribution

Our model generated:

```text
the cat sat on the mat
```

correctly from contexts it had seen during training.

But the next context became:

```text
["on", "the", "mat"]
```

which had never appeared as a training input.

Prediction confidence dropped and generation quickly became less coherent.

This demonstrated:

```text
low training loss
≠
good free-running generation
```

A model must generalize not only to held-out examples but also to contexts created by its own previous outputs.

---

## 5. Sampling preserves uncertainty

Instead of taking the argmax, we can sample according to the learned distribution.

For:

```text
mat ≈ 0.497
rug ≈ 0.497
```

10,000 samples produced empirical frequencies very close to those probabilities.

Therefore:

```text
probability distribution
→ deterministic for fixed model/context

sample drawn from distribution
→ stochastic
```

Different generation runs can therefore follow different trajectories.

---

## 6. Temperature reshapes the decoding distribution

Temperature modifies logits before softmax:

$$
P_i(T)
=
\frac{\exp(z_i/T)}
{\sum_j \exp(z_j/T)}
$$

The main behavior is:

```text
T < 1
→ sharper distribution

T = 1
→ original distribution

T > 1
→ flatter distribution
```

Low temperature suppresses weak alternatives.

High temperature makes lower-scoring tokens easier to sample.

Temperature does not change model parameters or teach the model anything new.

It changes how the model's existing logits are used during decoding.

---

## 7. Temperature cannot invent preferences

The model learned almost identical logits for `"mat"` and `"rug"`.

Even at:

```text
T = 0.01
```

their probabilities remained approximately:

```text
mat ≈ 0.504
rug ≈ 0.496
```

Temperature amplifies differences already present in the logits.

It does not create a strong preference where the model learned almost none.

---

## 8. Greedy decoding and temperature approaching zero are related but distinct

Greedy decoding explicitly performs:

```text
argmax(logits)
```

Temperature-based sampling instead computes:

```text
softmax(logits / T)
```

and samples from the resulting distribution.

As:

$$
T \rightarrow 0^+
$$

the probability distribution becomes increasingly concentrated around the maximum logit.

With a clear winner, this approaches greedy behavior.

But `T = 0` itself is not valid in the softmax temperature equation because it would require division by zero.

---

## Final Mental Model

The complete language-model pipeline developed across Module 3 is now:

```text
raw text
↓
tokens
↓
vocabulary
↓
token IDs
↓
embeddings
↓
context representation
↓
vocabulary logits
↓
cross-entropy training
↓
learned conditional distribution
↓
decoding
↓
generated token
↓
append to context
↓
repeat
```

The core idea is:

$$
P(\text{next token}\mid\text{context})
$$

Training learns this conditional distribution.

Decoding determines how we turn that distribution into an actual sequence.

The next step is to replace our crude flattened-context model with a mechanism that can represent interactions between token positions much more effectively.

That leads to **self-attention and Transformers**.